# AttentiveFP Molecular Property Prediction

Graph Property Prediction on MoleculeNet: Molecular property prediction with Attentive Fingerprint graph neural network. This notebook implements the approach with `AttentiveFP`, trained with the Adam optimizer, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `AttentiveFP` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

import os
import shutil

# Choose: "tensorflow", "torch", or "jax"
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import keras
from k3_node.datasets import MoleculeNet
from k3_node.loader import DataLoader
from k3_node.models import AttentiveFP

backend = keras.config.backend()
print(f"[K3-Node] Training AttentiveFP on Keras 3 ({backend})...")

# 1. Dataset (use force_reload=False if previously processed under another backend)
path = os.path.join(".", "data", "MoleculeNet")
dataset = MoleculeNet(path, name="ESOL", force_reload=False)

train_dataset = dataset[:800]
val_dataset = dataset[800:900]
test_dataset = dataset[900:]

BATCH_SIZE = 64
is_jax = backend == "jax"
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=is_jax)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, drop_last=is_jax)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, drop_last=is_jax)

# 2. Infinite Generator yielding clean NumPy arrays across all epochs
def to_np(t, dtype=None):
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader, infinite=True):
    def gen():
        while True:
            for batch in loader:
                inputs = {
                    "x": to_np(batch.x, dtype=np.float32),
                    "edge_index": to_np(batch.edge_index, dtype=np.int64),
                    "edge_attr": to_np(batch.edge_attr, dtype=np.float32),
                    "batch": to_np(batch.batch, dtype=np.int64),
                }
                yield inputs, to_np(batch.y, dtype=np.float32)
            if not infinite:
                break
    return gen

# 3. Model
out_channels = dataset[0].y.shape[-1]
model = AttentiveFP(
    in_channels=dataset.num_features,
    hidden_channels=64,
    out_channels=out_channels,
    edge_dim=dataset.num_edge_features,
    num_layers=3,
    num_timesteps=2,
    dropout=0.2,
    batch_size=BATCH_SIZE if is_jax else None,
)

# 4. Compile (jit_compile=False prevents XLA recompilations on variable graph sizes)
compile_kwargs = {}
if backend == "tensorflow":
    compile_kwargs["jit_compile"] = False

model.compile(
    optimizer=keras.optimizers.Adam(10**-2.5, weight_decay=10**-5),
    loss="mse",
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
    **compile_kwargs,
)

# 5. Fit across all epochs
epochs=5
history = model.fit(
    make_generator(train_loader, infinite=True)(),
    steps_per_epoch=len(train_loader),
    validation_data=make_generator(val_loader, infinite=True)(),
    validation_steps=len(val_loader),
    epochs=epochs,
    verbose=1,
)

# 6. Evaluate on Test Set
results = model.evaluate(
    make_generator(test_loader, infinite=False)(),
    steps=len(test_loader),
    verbose=1,
)
print(f"Test RMSE: {results[1]:.4f}")
